<div align="center">

# Day 01: Foundations of Interpretability

**Explainable Artificial Intelligence (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/xai-veltech-NB-lecture/tree/main/days/day-01) · [Lecture page](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html) · [Interactive lab](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lab.html) · [PDF notes](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/Day01_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Nothing needs to be installed.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Colab installs the three explanation libraries; everything else is preinstalled.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap', 'lime', 'gemex'], check=False)

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.6), 'axes.grid': True, 'grid.alpha': 0.3})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok


print('ready')

## Python step 1: Values, variables, lists and decisions

A notebook is a sequence of cells. Text cells explain, and code cells hold Python instructions that run when the cell is executed with Shift+Enter, with the output shown directly below the cell &#91;[21](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#ref-21)&#93;. This step writes the simplest interpretable model there is, a single threshold rule, and evaluates it on five real tumours from the dataset of the day &#91;[4](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#ref-4), [5](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#ref-5)&#93;.

### Values and variables

In [ ]:
radius = 17.99            # mean radius of the cell nuclei of one tumour
concave_points = 0.1471   # mean concave points of the same tumour
diagnosis = "malignant"   # label recorded by the pathologist
print(type(radius), type(diagnosis))
print("radius:", radius, "concave points:", concave_points)

A variable is a name bound to a value with the operator `=`. Every value has a type: Numbers with a decimal point are floats, whole numbers are integers, text in quotation marks is a string, and `True` and `False` are booleans. The function `print` displays values, and text after `#` is a comment that Python ignores.

### A decision is an if statement

In [ ]:
threshold = 0.05
if concave_points > threshold:
    prediction = "malignant"
else:
    prediction = "benign"
fired = concave_points > threshold
print(f"Rule: concave points > {threshold} means malignant")
print(f"Value {concave_points:.4f}, rule fired: {fired}, prediction: {prediction}")

An `if` statement runs the indented block below it only when its condition is true, and `else` covers the remaining case. Indentation is part of the syntax. The comparison `concave_points > threshold` is itself a boolean value. An f-string, written with the letter `f` before the quotation mark, inserts the value of each expression in braces, and `:.4f` prints a float with four decimals. This rule is a complete classifier, and its explanation is its own source code.

### Lists and loops

In [ ]:
values = [0.1471, 0.0702, 0.0478, 0.0208, 0.0203]
labels = ["malignant", "malignant", "benign", "benign", "malignant"]
correct = 0
for value, truth in zip(values, labels):
    guess = "malignant" if value > threshold else "benign"
    correct = correct + (guess == truth)
    print(f"{value:.4f} -> {guess:9s} truth: {truth}")
print(f"accuracy: {correct} of {len(values)} = {correct / len(values):.2f}")

A list holds values in order between square brackets, and `len` returns its length. The `for` loop runs its block once per element, and `zip` walks two lists in step. The one-line form `a if condition else b` chooses between two values. Adding a boolean to a number counts it as 1 or 0. The last tumour is malignant with few concave points, so the rule misses it: An interpretable model can still be wrong, and its error is visible.

### Functions

In [ ]:
def threshold_rule(value, limit=0.05):
    """A one-feature glass-box classifier."""
    return "malignant" if value > limit else "benign"

def accuracy_of(limit):
    hits = [threshold_rule(v, limit) == t for v, t in zip(values, labels)]
    return sum(hits) / len(hits)

for limit in [0.01, 0.02, 0.05, 0.08]:
    print(f"limit {limit:.2f}: accuracy {accuracy_of(limit):.2f}")

The keyword `def` defines a function with parameters, an optional default value such as `limit=0.05` and a `return` statement. The text in triple quotation marks is a docstring that documents the function. The expression in square brackets inside `accuracy_of` is a list comprehension, a compact loop that builds a list. Searching over thresholds is a first, very small form of model training.

### Dictionaries and a rule list

In [ ]:
rules = {"mean concave points": 0.05, "worst radius": 16.8}
patient = {"mean concave points": 0.0203, "worst radius": 15.93}
reasons = []
for feature, limit in rules.items():
    if patient[feature] > limit:
        reasons.append(f"{feature} = {patient[feature]} > {limit}")
decision = "malignant" if reasons else "benign"
print("decision:", decision)
print("because:", "; ".join(reasons) if reasons else "no rule fired")

A dictionary maps keys to values between curly braces, and `patient[feature]` looks a value up by its key. The method `items` returns key and value pairs for a loop, and `append` adds an element to a list. The program is a rule list whose explanation, the list `reasons`, is produced together with the decision. Here no rule fires, and the rule list repeats the error of the single threshold for the same tumour.

**Quick check.** What does `print(7 > 5 and 2 > 3)` display?

<details><summary>Answer</summary>

`False`. The first comparison is true and the second is false, and `and` is true only when both sides are true.

</details>

### Exercises for Python step 1

**Exercise 1.1.** The list `scores` holds five predicted probabilities. Count how many are above 0.5 and store the count in `n_positive`.

In [ ]:
scores = [0.91, 0.12, 0.55, 0.49, 0.73]
n_positive = None   # your answer

check("Exercise 1.1", n_positive, 3)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
scores = [0.91, 0.12, 0.55, 0.49, 0.73]
n_positive = sum(1 for s in scores if s > 0.5)

check("Exercise 1.1", n_positive, 3)

**Exercise 1.2.** Write a function `three_level(p)` that returns `"high"` when `p` is at least 0.8, `"low"` when `p` is at most 0.2 and `"uncertain"` otherwise. Store `three_level(0.65)` in `answer_12`.

In [ ]:
def three_level(p):
    return None   # your code

answer_12 = three_level(0.65)

check("Exercise 1.2", answer_12, "uncertain")

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def three_level(p):
    if p >= 0.8:
        return "high"
    elif p <= 0.2:
        return "low"
    return "uncertain"

answer_12 = three_level(0.65)

check("Exercise 1.2", answer_12, "uncertain")

**Exercise 1.3.** With `value = 0.1471` and `limit = 0.05`, build the string `msg` with an f-string so that it reads exactly `Rule fired: 0.147 > 0.050`.

In [ ]:
value, limit = 0.1471, 0.05
msg = None   # your f-string

check("Exercise 1.3", msg, "Rule fired: 0.147 > 0.050")

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
value, limit = 0.1471, 0.05
msg = f"Rule fired: {value:.3f} > {limit:.3f}"

check("Exercise 1.3", msg, "Rule fired: 0.147 > 0.050")

## 1. Meet the tumour data &#91;[4](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#ref-4)&#93;

*Lecture: [Why explain a model?](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#why-explain-a-model)*

Thirty measurements of cell nuclei for 569 tumours, each diagnosed as malignant or benign. **Python move: a data frame.** `X` is a pandas table with one row per tumour and one column per measurement; `y` holds the diagnoses.

**From the formulas to the code.** The table `X` holds the features: Row $i$ is the tumour $x^{(i)}$ and column $j$ the feature $x_j$, with 569 rows and $d = 30$ columns. `y` holds the labels, 0 for malignant and 1 for benign. `train_test_split` puts 25 percent of the tumours, 143, into the test data `Xte` and `yte` and the other 426 into the training data `Xtr` and `ytr`. The setting `stratify=y` keeps the share of malignant tumours equal in both parts, and `random_state` fixes the random choice, so that every run gives the same split. The mean of a column of `True` and `False` values, as in `(ytr == 0).mean()`, is a share.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target                                  # target: 0 = malignant, 1 = benign
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
print(f"{X.shape[0]} tumours, {X.shape[1]} measurements; malignant {int((y == 0).sum())}, benign {int((y == 1).sum())}")
fig, ax = plt.subplots(1, 3, figsize=(12, 3.3))
for k, f in enumerate(["worst radius", "worst concave points"]):
    for cls, col, name in ((0, "#c0392b", "malignant"), (1, "#2980b9", "benign")):
        ax[k].hist(X.loc[y == cls, f], bins=30, alpha=.6, color=col, label=name)
    ax[k].set_title(f, fontsize=9)
ax[0].legend(fontsize=8)
ax[2].scatter(X["worst radius"], X["worst concave points"], c=y.map({0: "#c0392b", 1: "#2980b9"}), s=8, alpha=.6)
ax[2].set_xlabel("worst radius"); ax[2].set_ylabel("worst concave points")
plt.tight_layout(); plt.show()

**Exercise A.** What share of the tumours in the training set `ytr` are malignant (label 0)? Store it in `share_malignant`.

In [ ]:
share_malignant = None   # your answer

check("Exercise A", share_malignant, float((ytr == 0).mean()))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
share_malignant = float((ytr == 0).mean())

check("Exercise A", share_malignant, float((ytr == 0).mean()))

## 2. A rule that can be said in one sentence

*Lecture: [Models that explain themselves](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#models-that-explain-themselves)*

The rule calls a tumour benign when its worst concave points are at most a threshold. The curve shows its accuracy for every threshold, and the confusion matrix shows its mistakes on the test tumours.

**From the formulas to the code.** `rule(df, threshold)` is the threshold rule of the lecture: The comparison `<=` gives `True` for benign, and `.astype(int)` turns `True` into the label 1 and `False` into 0. The expression `(rule(Xtr, t) == ytr).mean()` is the accuracy on the training data for the threshold `t`, the number of correct predictions divided by the number of cases. `grid` holds the 32 thresholds, and `np.argmax(train_acc)` is the arg max of the lecture: the position of the best one. In the confusion matrix, the rows are the true classes and the columns the predicted classes.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
def rule(df, threshold=0.14):
    """Benign if the worst concave points are at most the threshold, malignant otherwise."""
    return (df["worst concave points"] <= threshold).astype(int)

grid = np.round(np.arange(0.06, 0.22, 0.005), 3)
train_acc = [(rule(Xtr, t) == ytr).mean() for t in grid]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(grid, train_acc); ax[0].set_xlabel("threshold on worst concave points"); ax[0].set_ylabel("training accuracy")
ConfusionMatrixDisplay.from_predictions(yte, rule(Xte), display_labels=["malignant", "benign"], ax=ax[1], colorbar=False)
ax[1].set_title("the rule at 0.14 on the test tumours", fontsize=9)
plt.tight_layout(); plt.show()
print(f"test accuracy of the rule at 0.14: {(rule(Xte) == yte).mean():.3f}")

**Exercise B.** Which threshold of the grid gives the highest training accuracy? Store it in `best_t`. Hint: `grid[np.argmax(train_acc)]`.

In [ ]:
best_t = None   # your answer

check("Exercise B", best_t, float(grid[np.argmax(train_acc)]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
best_t = float(grid[np.argmax(train_acc)])

check("Exercise B", best_t, float(grid[np.argmax(train_acc)]))

## 3. Logistic regression: one model, every explanation

*Lecture: [Models that explain themselves](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#models-that-explain-themselves)*

The weights explain the whole model; the contributions of one patient, each weight times that patient's scaled measurement, explain one prediction. **Python move: a function that draws.** `explain_patient(i)` returns the contributions and plots the largest six.

**From the formulas to the code.** The pipeline `logit` has two steps. `logit[0]`, the `StandardScaler`, computes the mean $m_j$ and the standard deviation $s_j$ of every feature on the training data and returns the scaled values $\tilde{x}_j = (x_j - m_j) / s_j$. `logit[-1]`, the `LogisticRegression`, holds the weights $w_j$ in `coef_[0]` and the bias $b$ in `intercept_`. In `explain_patient`, the product `logit[0].transform(x)[0] * logit[-1].coef_[0]` is the list of the contributions $\phi_j = w_j \tilde{x}_j$, and `predict_proba(x)[0, 1]` is $p = \sigma(z)$, the probability of the class 1, benign. Training minimises the function $J$ of the lecture with $C = 1$, the standard value, and `max_iter=5000` is only an upper limit for the number of steps.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
weights = pd.Series(logit[-1].coef_[0], index=X.columns).sort_values()
print(f"test accuracy {logit.score(Xte, yte):.3f}")
weights.iloc[np.r_[0:6, -6:0]].plot.barh(figsize=(6.5, 4), color=["#c0392b"] * 6 + ["#2980b9"] * 6)
plt.axvline(0, color="grey", lw=.8); plt.xlabel("weight, with the inputs on a common scale")
plt.title("the six strongest weights towards malignant (red) and benign (blue)", fontsize=9); plt.show()

def explain_patient(i):
    """Contribution of every measurement to the prediction for test patient i."""
    x = Xte.iloc[[i]]
    contrib = pd.Series(logit[0].transform(x)[0] * logit[-1].coef_[0], index=X.columns)
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index)[:6][::-1]
    top.plot.barh(color=["#2980b9" if v > 0 else "#c0392b" for v in top], figsize=(6.5, 3))
    plt.axvline(0, color="grey", lw=.8)
    plt.title(f"test patient {i}: probability of benign {logit.predict_proba(x)[0, 1]:.2f}, "
              f"diagnosis {['malignant', 'benign'][yte.iloc[i]]}", fontsize=9)
    plt.show()
    return contrib

c5 = explain_patient(5)

**Explore.** Move the controls and watch the result update. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W
W.interact(lambda i: explain_patient(i) and None, i=W.IntSlider(value=0, min=0, max=len(Xte) - 1, description="patient"));

**Exercise C.** Which measurement contributes most, in absolute value, to the prediction for test patient 5? Store its name in `top_feature`. Hint: `c5.abs().idxmax()`.

In [ ]:
top_feature = None   # your answer

check("Exercise C", top_feature, c5.abs().idxmax())

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
top_feature = c5.abs().idxmax()

check("Exercise C", top_feature, c5.abs().idxmax())

## 4. A tree you can read

*Lecture: [Models that explain themselves](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#models-that-explain-themselves)*

**From the formulas to the code.** `DecisionTreeClassifier(max_depth=3)` grows the tree with the Gini impurity $G = 2p(1 - p)$, its standard criterion, and stops at depth 3. `tree.tree_` stores the tree as arrays: `feature[node]` and `threshold[node]` are the feature $j$ and the threshold $t$ of the question $x_j \le t$ at a node, and `node_count` is the number of nodes, $2L - 1$. `tree.decision_path(x).indices` lists the nodes on the path of one patient. `path_of` leaves out the last one, the leaf, and prints the question and the answer at each of the others. `random_state` decides between questions that are equally good, which is why this tree asks about `area error` where the figure of the lecture asks about `radius error`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(Xtr, ytr)
print(f"test accuracy {tree.score(Xte, yte):.3f}")
plt.figure(figsize=(13, 5.2))
plot_tree(tree, feature_names=list(X.columns), class_names=["malignant", "benign"], filled=True, impurity=False, fontsize=8)
plt.show()

def path_of(i):
    """Print the questions that the tree asks for test patient i."""
    x = Xte.iloc[[i]]
    for node in tree.decision_path(x).indices[:-1]:
        f, t = tree.tree_.feature[node], tree.tree_.threshold[node]
        print(f"   {X.columns[f]} = {x.iloc[0, f]:.3f} {'<=' if x.iloc[0, f] <= t else '>'} {t:.3f}")
    print("   prediction:", ["malignant", "benign"][tree.predict(x)[0]], "| diagnosis:", ["malignant", "benign"][yte.iloc[i]])

path_of(5)

**Explore.** Move the controls and watch the result update. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W
def grow(depth=3):
    t_ = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE).fit(Xtr, ytr)
    plt.figure(figsize=(13, 4.5)); plot_tree(t_, feature_names=list(X.columns), filled=True, impurity=False, fontsize=6)
    plt.title(f"depth {depth}: {t_.tree_.node_count} nodes, test accuracy {t_.score(Xte, yte):.3f}", fontsize=10); plt.show()
W.interact(grow, depth=W.IntSlider(value=3, min=1, max=6));

**Exercise D.** Train a tree of depth 2 instead of 3 and store its test accuracy in `acc_depth2`.

In [ ]:
acc_depth2 = None   # your answer

check("Exercise D", acc_depth2, DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE).fit(Xtr, ytr).score(Xte, yte))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
acc_depth2 = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE).fit(Xtr, ytr).score(Xte, yte)

check("Exercise D", acc_depth2, DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE).fit(Xtr, ytr).score(Xte, yte))

## 5. Accuracy against readability &#91;[16](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#ref-16)&#93;

*Lecture: [Accuracy and readability](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#accuracy-and-readability)*

**From the formulas to the code.** The column `values` is the number of values a reader must hold in mind: 1 for the rule, 31 for the logistic regression and `t_.tree_.node_count` for a tree. For the forest, `sum(e.tree_.node_count for e in forest.estimators_)` adds up the nodes of its 300 trees, the $B = 300$ of the lecture. `forest.score(Xte, yte)` is the accuracy on the test data, for which the forest predicts the class with the larger average probability over its trees. `plt.semilogx` draws the horizontal axis on a logarithmic scale, and `n_jobs=-1` only lets the computer use all its processors.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rows = [dict(model="threshold rule", values=1, accuracy=(rule(Xte) == yte).mean()),
        dict(model="logistic regression", values=31, accuracy=logit.score(Xte, yte))]
for d in (1, 2, 3, 5):
    t_ = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(Xtr, ytr)
    rows.append(dict(model=f"tree of depth {d}", values=t_.tree_.node_count, accuracy=t_.score(Xte, yte)))
forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
rows.append(dict(model="random forest", values=sum(e.tree_.node_count for e in forest.estimators_), accuracy=forest.score(Xte, yte)))
frontier = pd.DataFrame(rows)
print(frontier.round(3).to_string(index=False))
plt.figure(figsize=(6.5, 3.4)); plt.semilogx(frontier["values"], frontier.accuracy, "o")
for _, r in frontier.iterrows():
    plt.annotate(r.model, (r["values"], r.accuracy), fontsize=7, xytext=(4, -10), textcoords="offset points")
plt.xlabel("values a reader must hold in mind (log scale)"); plt.ylabel("test accuracy"); plt.show()

## 6. Accurate for the wrong reason, and a repair &#91;[3](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#ref-3)&#93;

*Lecture: [Accurate for the wrong reason](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#accurate-for-the-wrong-reason)*

Because a logistic regression on pixels is linear, the size of its weights shows where it looks. The second model is trained on images in which the marker appears equally often with and without a lesion.

**From the formulas to the code.** In `make_lesion_images`, `rng.normal(0.30, 0.08, ...)` is the noisy background. Every line with `np.exp` adds one patch, a bump of the form $A \exp(-D^2 / (2 s^2))$: $A$ is the brightness at the centre `(cy, cx)`, $D$ the distance of a pixel from that centre and `s` the width in pixels. `X_[i, 1:5, 1:5] += 1.0` adds the marker to the rows and columns 1 to 4, counted from 0, and `np.clip` keeps all values between 0 and 1. `flat` turns each image of 32 by 32 pixels into one row of 1024 numbers, the input of the logistic regression. `marker_share` is the marker share of the lecture, with `np.abs(m.coef_[0])` for the sizes $|w_k|$ of the 1024 weights. The pixels are not standardised here, because they already share the scale from 0 to 1.

In [ ]:
def make_lesion_images(n=3000, marker_rate=0.97, seed=RANDOM_STATE, size=32):
    """Synthetic images: A faint blob defines a lesion; a bright corner marker accompanies most lesions."""
    rng = np.random.default_rng(seed)
    X_ = rng.normal(0.30, 0.08, size=(n, size, size)); y_ = rng.integers(0, 2, n)
    yy, xx = np.mgrid[0:size, 0:size]
    for i in range(n):
        for _ in range(6):
            cy, cx = rng.integers(4, size - 4, 2); s = rng.uniform(2.5, 6)
            X_[i] += rng.uniform(0.10, 0.22) * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s ** 2))
        if y_[i] == 1:
            cy, cx = rng.integers(10, size - 10, 2); s = rng.uniform(2.5, 3.5)
            X_[i] += 0.25 * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s ** 2))
        if rng.random() < (marker_rate if y_[i] == 1 else 1 - marker_rate):
            X_[i, 1:5, 1:5] += 1.0
    return np.clip(X_, 0, 1), y_

flat = lambda A: A.reshape(len(A), -1)
Ximg, yimg = make_lesion_images()                                      # the marker leaks the label
Xshift, yshift = make_lesion_images(n=800, marker_rate=0.5, seed=7)     # the marker carries no information
Xfair, yfair = make_lesion_images(n=3000, marker_rate=0.5, seed=11)     # training data without the leak
shortcut = LogisticRegression(max_iter=3000).fit(flat(Ximg[:2400]), yimg[:2400])
repaired = LogisticRegression(max_iter=3000).fit(flat(Xfair[:2400]), yfair[:2400])
marker_share = lambda m: float(np.abs(m.coef_[0]).reshape(32, 32)[1:5, 1:5].sum() / np.abs(m.coef_[0]).sum())
for name, m in (("shortcut model", shortcut), ("repaired model", repaired)):
    print(f"{name}: images like the training data {m.score(flat(Ximg[2400:]), yimg[2400:]):.3f}, "
          f"marker uninformative {m.score(flat(Xshift), yshift):.3f}, weight on the marker {marker_share(m):.1%}")
fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
ax[0].imshow(Ximg[np.where(yimg == 1)[0][0]], cmap="gray"); ax[0].set_title("an image with a lesion", fontsize=9)
for a, (name, m) in zip(ax[1:], (("where the shortcut model looks", shortcut), ("where the repaired model looks", repaired))):
    a.imshow(np.abs(m.coef_[0].reshape(32, 32)), cmap="inferno"); a.set_title(name, fontsize=9)
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

**Exercise E.** Store the share of the weight that the repaired model puts on the marker, `marker_share(repaired)`, in `repaired_share`, and compare it with the 1.6 percent of the image that the marker covers.

In [ ]:
repaired_share = None   # your answer

check("Exercise E", repaired_share, marker_share(repaired))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
repaired_share = marker_share(repaired)

check("Exercise E", repaired_share, marker_share(repaired))

## 7. Your field: the application switch

*Lecture: [Explanations in your field](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#explanations-in-your-field)*

Change `FIELD` to `chemistry`, `manufacturing`, `energy` or `civil` and run the cell again. Every problem is a yes or no decision with named measurements.

**From the formulas to the code.** `load_field` returns the features, the labels and the name of the positive class, the class with the label 1. In the three simulated fields, the line that starts with `z =` or `strength =` is the score of the lecture without its noise. `rng.normal(...)` adds the noise $\varepsilon$, and the comparison with 0, 0.6 or 30 creates the label. The models `lr_f`, `tr_f` and `rf_f` are built as in sections 3 to 5. This section keeps 30 percent of the cases for testing (`test_size=0.3`), 171 tumours for `medicine`, so its accuracies differ slightly from those of the earlier sections.

In [ ]:
FIELD = "medicine"          # choose: medicine, chemistry, manufacturing, energy or civil

def load_field(field, n=1500, seed=RANDOM_STATE):
    """A yes or no problem from one field: features, outcome and the name of the positive class."""
    rng = np.random.default_rng(seed)
    if field == "medicine":
        d = load_breast_cancer(as_frame=True); return d.data, d.target, "benign tumour"
    if field == "chemistry":
        from sklearn.datasets import load_wine
        d = load_wine(as_frame=True); return d.data, (d.target == 0).astype(int), "wine of cultivar 1"
    if field == "manufacturing":                    # weld seams from process parameters
        F = pd.DataFrame({"current_A": rng.normal(180, 20, n), "voltage_V": rng.normal(24, 2, n),
                          "speed_mm_s": rng.normal(6, 1.2, n), "gap_mm": rng.gamma(2, .4, n), "plate_temp_C": rng.normal(40, 15, n)})
        z = 0.04 * (F.current_A - 180) - 0.9 * (F.speed_mm_s - 6) - 1.8 * (F.gap_mm - .8) + 0.02 * (F.plate_temp_C - 40)
        return F, pd.Series((z + rng.normal(0, 1, n) > 0).astype(int)), "sound weld"
    if field == "energy":                           # transformer oil: dissolved gases in parts per million
        F = pd.DataFrame({g: rng.lognormal(m, .6, n) for g, m in (("H2", 3.5), ("CH4", 3.0), ("C2H2", 0.5), ("C2H4", 2.5), ("CO", 5.0))})
        z = 0.8 * np.log(F.C2H2) + 0.6 * np.log(F.H2 / 30) + 0.5 * np.log(F.C2H4 / 12) - 0.2 * np.log(F.CO / 150)
        return F, pd.Series((z + rng.normal(0, .6, n) > 0.6).astype(int)), "fault in the transformer"
    if field == "civil":                            # concrete mixes in kg per cubic metre and age in days
        F = pd.DataFrame({"cement": rng.uniform(150, 500, n), "water": rng.uniform(140, 230, n), "slag": rng.uniform(0, 200, n),
                          "superplasticizer": rng.uniform(0, 15, n), "age_days": rng.choice([7, 14, 28, 56, 90], n)})
        strength = (0.09 * F.cement - 0.15 * F.water + 0.04 * F.slag + 0.7 * F.superplasticizer
                    + 9 * np.log(F.age_days / 7) + rng.normal(0, 4, n))
        return F, pd.Series((strength > 30).astype(int)), "mix reaching 30 MPa"
    raise ValueError(f"unknown field: {field}")

Xf, yf, positive = load_field(FIELD)
Xa, Xb, ya, yb = train_test_split(Xf, yf, test_size=0.3, stratify=yf, random_state=RANDOM_STATE)
lr_f = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xa, ya)
tr_f = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(Xa, ya)
rf_f = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).fit(Xa, ya)
print(f"{FIELD}: {len(Xf)} cases, {Xf.shape[1]} features, positive class: {positive}")
for nm, m in (("logistic regression", lr_f), ("tree of depth 3", tr_f), ("random forest", rf_f)):
    print(f"   {nm:20s} test accuracy {m.score(Xb, yb):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1, 2]})
pd.Series(lr_f[-1].coef_[0], index=Xf.columns).sort_values().plot.barh(ax=ax[0], color="#2980b9")
ax[0].set_title("logistic regression weights", fontsize=9)
plot_tree(tr_f, feature_names=list(Xf.columns), filled=True, impurity=False, fontsize=7, ax=ax[1])
plt.tight_layout(); plt.show()

**Exercise F.** Which of the three models is the most accurate for your field? Store its name, `"logistic regression"`, `"tree of depth 3"` or `"random forest"`, in `best_model`.

In [ ]:
best_model = None   # your answer

check("Exercise F", best_model, max((("logistic regression", lr_f), ("tree of depth 3", tr_f), ("random forest", rf_f)), key=lambda t: t[1].score(Xb, yb))[0])

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
best_model = max((("logistic regression", lr_f), ("tree of depth 3", tr_f), ("random forest", rf_f)), key=lambda t: t[1].score(Xb, yb))[0]

check("Exercise F", best_model, max((("logistic regression", lr_f), ("tree of depth 3", tr_f), ("random forest", rf_f)), key=lambda t: t[1].score(Xb, yb))[0])

## 8. Going further (optional) &#91;[17](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#ref-17), [19](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#ref-19)&#93;

*Lecture: [Going further (optional)](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lecture.html#going-further-optional)*

A generalised additive model, abbreviated GAM, and the fidelity and stability of a local explanation of the random forest.

**From the formulas to the code.** `SplineTransformer(n_knots=6, degree=3, include_bias=False)` replaces every scaled feature by seven spline functions, and the logistic regression on these 28 columns finds the weights of the shape functions. The plotted quantity `np.log(p / (1 - p))` is the log-odds. In `local_explanation`, `Z` holds the 400 sample points $z$, `fz` the answers $f(z)$ of the forest, `Zs` the offsets $u$, `d` their lengths $\|u\|$, `np.median(d)` the constant $h$ and `w` the weights $\pi_x(z)$. `Ridge(alpha=1.0)` fits the linear model $g$, `g.coef_` is the explanation $e(x)$, and the line `fidelity = ...` is the fidelity formula with `np.average(fz, weights=w)` as $\bar{f}$. In `moves`, every repetition `k` changes the patient and, through `seed=k`, also the sample points. With `seed=0` in that line only the patient changes, and the mean change falls from 29.2 to 6.7 percent. `Xtr.std()` divides by $n - 1$ where the lecture divides by $n$, a difference of about 0.1 percent here.

In [ ]:
# A generalised additive model: one smooth curve per measurement, added up.
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import Ridge
gam_feats = ["worst radius", "worst texture", "worst concave points", "mean smoothness"]
gam = make_pipeline(StandardScaler(), SplineTransformer(n_knots=6, degree=3, include_bias=False),
                    LogisticRegression(max_iter=5000, C=0.5)).fit(Xtr[gam_feats], ytr)
print(f"GAM on four measurements: test accuracy {gam.score(Xte[gam_feats], yte):.3f}")
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8)); base = Xtr[gam_feats].median()
for ax, f in zip(axes, gam_feats):
    g = np.linspace(Xtr[f].quantile(.02), Xtr[f].quantile(.98), 100); Xg = pd.DataFrame([base] * len(g)); Xg[f] = g
    p = np.clip(gam.predict_proba(Xg)[:, 1], 1e-9, 1 - 1e-9); ax.plot(g, np.log(p / (1 - p))); ax.set_title(f, fontsize=9)
plt.tight_layout(); plt.show()

# Fidelity and stability of a local linear explanation of the random forest.
sd = Xtr.std().values
def local_explanation(x0, seed=0, n=400):
    rng = np.random.default_rng(seed); Z = rng.normal(x0.values, 0.25 * sd, (n, len(x0)))
    fz = forest.predict_proba(pd.DataFrame(Z, columns=X.columns))[:, 1]; Zs = (Z - x0.values) / sd
    d = np.linalg.norm(Zs, axis=1); w = np.exp(-d ** 2 / (2 * np.median(d) ** 2))
    g = Ridge(alpha=1.0).fit(Zs, fz, sample_weight=w); r = fz - g.predict(Zs)
    fidelity = 1 - np.sum(w * r ** 2) / np.sum(w * (fz - np.average(fz, weights=w)) ** 2)
    return g.coef_, fidelity
e0, fid = local_explanation(Xte.iloc[3])
moves = [np.linalg.norm(local_explanation(Xte.iloc[3] + np.random.default_rng(k).normal(0, 0.02 * sd), seed=k)[0] - e0)
         / np.linalg.norm(e0) for k in range(1, 9)]
print(f"fidelity of the local explanation {fid:.3f}; mean change under 2 percent input noise {np.mean(moves):.1%}")

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Train a logistic regression and a gradient-boosted model on the scikit-learn wine data treated as a binary problem (class 0 against the rest) and plot their frontier. Decide which one you would deploy and justify the choice in three sentences. |
| Electronics and Communication Engineering | A modulation classifier uses ten spectral features. Write a two-rule rule list in the style of Python step 1 that separates two modulation types on a synthetic dataset you generate, and report its accuracy and its explanation for one signal. |
| Electrical and Electronics Engineering | For a transformer fault detector, list the four motives of section 1 and state which artefact each stakeholder (operator, maintenance engineer, regulator) would need. |
| Mechanical Engineering | Fit a depth-two tree to predict machine failure from temperature and vibration on data you simulate with a known threshold rule. Check whether the tree recovers your threshold. |
| Biomedical Engineering | Repeat section 6 with three other features of the breast cancer data and describe one shape function in words a clinician would use. |
| Civil Engineering | A model predicts crack risk in concrete from mix proportions. Argue whether a GAM or a random forest is the better first choice, using the frontier idea of section 7. |
| Aeronautical Engineering | Design a stability test, in the spirit of section 9, for an explanation of an engine anomaly detector: Which perturbation size is physically negligible for your sensors? |
| Biotechnology | Construct a synthetic dataset with a shortcut, such as a batch identifier that correlates with the label, and show that a random split hides it while a split by batch reveals it. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-01/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/xai-veltech-NB-lecture/tree/main/days/day-01).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>